# ML-09 -- Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haideriqbal499/Flyrank_Ml_internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits Week-5 (Lane 2 refresh scoring) the way the live session audited
FlyRank's published starter research numbers -- methodology questions first, then an honest
split, a leakage hunt, failure examples, and claim language that stays inside the evidence.

> Skills: `hunting-leakage-and-validating` + `flyrank/flyrank-data`.

## 1. Two paper findings + my methodology questions

Source: FlyRank's verified starter research results (`outputs/model_report.md` /
Hugging Face internship-starter reference numbers) -- the same disclosure the live session walked.

### Finding A -- Learned ranking beats the rule baseline by ~3x at Precision@50

**Published number:** rule baseline Precision@50 ~ 0.24 -> random forest Precision@50 ~ 0.74
on the 30k anonymized starter slice (client-holdout).

**Methodology question I would ask (constructively):**
Where does the label come from? `is_declining_label` is `trend_direction == "down"`, and
`trend_direction` is computed from `trend_pct` inside the *same* snapshot -- not a sealed
next-30-day outcome. Does Precision@50 on that contemporaneous proxy support a claim about
"pages an editor should refresh first," or only "pages that already look down-trending in
this window"? A stronger next step would keep the same ranking metric but redefine the label
as a true past->future window so the validation design matches the product decision.

### Finding B -- Client-holdout is presented as evidence the ranking transfers

**Published design:** whole clients held out of training; metrics reported on unseen clients.

**Methodology question I would ask (constructively):**
Does the validation design carry a transfer claim? With only 32 clients and ~20% held out
(~6 test clients), the Precision@50 estimate can swing with which clients land in the
holdout. Reporting the random-split number next to the grouped number -- and, later, a
time-aware past->future split on the warehouse -- would show how much of the lift is
client memorization versus a portable ranking idea. That is how to make the claim stronger,
not how to dismiss it.

In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/haideriqbal499/Flyrank_Ml_internship"
REPO_DIR = "Flyrank_Ml_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.abspath(os.sep):
        os.chdir("..")

CSV = Path("data/raw/content_refresh_anonymized.csv")
OUT_DIR = Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_PATH = OUT_DIR / "w06_validation_audit.json"

# Reproduce the published paper numbers from the committed starter report (no private data).
report_md = Path("outputs/model_report.md")
assert report_md.exists(), "Expected outputs/model_report.md (FlyRank starter research receipt)"
text = report_md.read_text(encoding="utf-8")
print("Published starter research numbers (from outputs/model_report.md):")
for line in text.splitlines():
    if "baseline_rules" in line or "random_forest" in line or "Split strategy" in line:
        print(" ", line.strip().lstrip("- ").lstrip("|").strip())

print("\nReference table used in the live-session walkthrough:")
print("  baseline_rules  Precision@50 = 0.240  | ROC AUC = 0.627")
print("  random_forest   Precision@50 = 0.740  | ROC AUC = 0.750")
print("  split = client_holdout | label = is_declining_label (trend_direction == down)")

print("\nMethodology focus for Finding A: label = snapshot trend proxy, not future window.")
print("Methodology focus for Finding B: client-holdout with n_clients=32 -- report gap vs random.")


Published starter research numbers (from outputs/model_report.md):
  Split strategy used for validation: client_holdout
  Best model: `random_forest` selected by `precision_at_50`.
  random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
  baseline_rules | 0.627 | 0.468 | 0.240 | - | - |

Reference table used in the live-session walkthrough:
  baseline_rules  Precision@50 = 0.240  | ROC AUC = 0.627
  random_forest   Precision@50 = 0.740  | ROC AUC = 0.750
  split = client_holdout | label = is_declining_label (trend_direction == down)

Methodology focus for Finding A: label = snapshot trend proxy, not future window.
Methodology focus for Finding B: client-holdout with n_clients=32 -- report gap vs random.


## 2. My model under an honest split (before/after)

Week-5 already used a **client holdout**. This section makes the honesty gain visible by
re-running the **same Random Forest + same features + same declining proxy** under two splits:

| Split | What it tests | Role here |
|---|---|---|
| **Before -- random row split** | Rows from the same client can appear in both train and test | Flattering / memorization-friendly |
| **After -- client holdout** | Ranking idea on clients the model never saw | Honest for multi-client portfolios |

The **gap** between the two numbers is itself a finding. Base rate is printed next to every metric.

*(Starter CSV has no calendar dates for a true time split -- client grouping is the honest
design available here. A warehouse past->future split remains future work.)*

In [2]:
df = pd.read_csv(CSV)
df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"].clip(lower=0))
df["log_clicks_90d"] = np.log1p(df["clicks_90d"].clip(lower=0))
df["log_sessions_90d"] = np.log1p(df["sessions_90d"].clip(lower=0))
df["has_position"] = (df["avg_position"] > 0).astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["word_count_filled"] = df["word_count"].fillna(0)

# Same feature set as Week-5 (no trend / no last-prev 30d windows)
NUMERIC_FEATURES = [
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "has_position",
    "engagement_rate",
    "scroll_rate",
    "word_count_filled",
    "has_word_count",
]

X_num = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce")
X_num = X_num.replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].astype(int)
clients = df["client_id"].fillna("unknown").astype(str)


def precision_at_k(scores: np.ndarray, labels: np.ndarray, k: int) -> float:
    order = np.argsort(-np.asarray(scores))
    top = np.asarray(labels)[order[:k]]
    return float(top.mean()) if len(top) else 0.0


def eval_rf(train_idx, test_idx, split_name: str) -> dict:
    X_tr, X_te = X_num.iloc[train_idx], X_num.iloc[test_idx]
    y_tr = y.iloc[train_idx].to_numpy()
    y_te = y.iloc[test_idx].to_numpy()
    assert np.unique(y_tr).size == 2 and np.unique(y_te).size == 2

    rf = RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf.fit(X_tr, y_tr)
    proba = rf.predict_proba(X_te)[:, 1]
    base_rate = float(y_te.mean())
    return {
        "split": split_name,
        "n_train": int(len(train_idx)),
        "n_test": int(len(test_idx)),
        "base_rate_test": round(base_rate, 4),
        "precision@10": round(precision_at_k(proba, y_te, 10), 4),
        "precision@50": round(precision_at_k(proba, y_te, 50), 4),
        "roc_auc": round(float(roc_auc_score(y_te, proba)), 4),
        "avg_precision": round(float(average_precision_score(y_te, proba)), 4),
        "rf": rf,
        "proba": proba,
        "y_te": y_te,
        "test_idx": test_idx,
    }


# --- BEFORE: random row split ---
train_rand, test_rand = train_test_split(
    np.arange(len(df)),
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)
before = eval_rf(train_rand, test_rand, "random_row_split")

# --- AFTER: client holdout (Week-5 honest design) ---
unique_clients = clients.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
n_test_clients = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test_clients])
test_mask = clients.isin(test_clients).to_numpy()
train_cli = np.where(~test_mask)[0]
test_cli = np.where(test_mask)[0]
after = eval_rf(train_cli, test_cli, "client_holdout")

comparison = pd.DataFrame(
    [
        {k: before[k] for k in ["split", "n_train", "n_test", "base_rate_test",
                                 "precision@10", "precision@50", "roc_auc", "avg_precision"]},
        {k: after[k] for k in ["split", "n_train", "n_test", "base_rate_test",
                                "precision@10", "precision@50", "roc_auc", "avg_precision"]},
    ]
)

print(f"Rows: {len(df):,} | clients: {clients.nunique()} | overall declining rate: {y.mean():.3f}")
print(f"Client holdout: train clients={len(unique_clients) - n_test_clients}, "
      f"test clients={n_test_clients}")
print("\nBefore / after (same RF, same features):")
print(comparison.to_string(index=False))

gap_p50 = before["precision@50"] - after["precision@50"]
gap_auc = before["roc_auc"] - after["roc_auc"]
print(f"\nGap (random - client_holdout): precision@50 = {gap_p50:+.3f}, roc_auc = {gap_auc:+.3f}")
print(
    "Reading: if the random split looks much better, some of that lift was client structure, "
    "not a portable ranking skill. Keep the client-holdout number for claims."
)


Rows: 30,000 | clients: 32 | overall declining rate: 0.542
Client holdout: train clients=26, test clients=6

Before / after (same RF, same features):
           split  n_train  n_test  base_rate_test  precision@10  precision@50  roc_auc  avg_precision
random_row_split    24000    6000           0.542           1.0          0.96   0.7714         0.7843
  client_holdout    27675    2325           0.391           0.9          0.90   0.7641         0.6684

Gap (random - client_holdout): precision@50 = +0.060, roc_auc = +0.007
Reading: if the random split looks much better, some of that lift was client structure, not a portable ranking skill. Keep the client-holdout number for claims.


## 3. Leakage audit

Attack checklist on the **final Week-5 feature set**, plus a deliberate leak test and
concrete failure examples on the honest (client-holdout) test set.

| Check | Verdict |
|---|---|
| Timeline / window overlap | Starter features are 90d snapshot aggregates; label is also snapshot-derived -- disclosed proxy risk, not a silent future leak |
| Label-derived / sibling columns | `trend_direction`, `trend_pct`, `is_declining_label` excluded from X |
| Product flags as features | Not present in starter CSV; not used |
| IDs as features | `client_id` / `content_id` used only for grouping / display |
| Split | Client holdout (honest number kept) |
| Base rate next to metrics | Printed in section 2 |
| Importance sanity | No single feature near 1.0 |

Deliberate attack: add `trend_pct` (label sibling) and watch the score jump -- then remove it.

In [3]:
# --- Leakage attack: honest features vs +trend_pct ---
X_honest = X_num.copy()
X_leaky = X_honest.copy()
X_leaky["trend_pct"] = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)


def score_on_holdout(X_frame: pd.DataFrame) -> dict:
    rf = RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf.fit(X_frame.iloc[train_cli], y.iloc[train_cli])
    proba = rf.predict_proba(X_frame.iloc[test_cli])[:, 1]
    y_te = y.iloc[test_cli].to_numpy()
    return {
        "precision@50": round(precision_at_k(proba, y_te, 50), 4),
        "roc_auc": round(float(roc_auc_score(y_te, proba)), 4),
        "top_feature": (
            pd.Series(rf.feature_importances_, index=X_frame.columns)
            .sort_values(ascending=False)
            .index[0]
        ),
        "top_importance": round(
            float(
                pd.Series(rf.feature_importances_, index=X_frame.columns)
                .sort_values(ascending=False)
                .iloc[0]
            ),
            4,
        ),
        "rf": rf,
        "proba": proba,
        "y_te": y_te,
    }


honest_scores = score_on_holdout(X_honest)
leaky_scores = score_on_holdout(X_leaky)

print("Leakage hunt (client-holdout test):")
print(
    f"  Honest features ({len(NUMERIC_FEATURES)}): "
    f"precision@50={honest_scores['precision@50']}, roc_auc={honest_scores['roc_auc']}, "
    f"top={honest_scores['top_feature']} ({honest_scores['top_importance']})"
)
print(
    f"  + trend_pct (label sibling): "
    f"precision@50={leaky_scores['precision@50']}, roc_auc={leaky_scores['roc_auc']}, "
    f"top={leaky_scores['top_feature']} ({leaky_scores['top_importance']})"
)
print(
    f"  Jump: roc_auc {leaky_scores['roc_auc'] - honest_scores['roc_auc']:+.4f}, "
    f"precision@50 {leaky_scores['precision@50'] - honest_scores['precision@50']:+.4f}"
)
print("  Keep the honest number; trend_pct stays excluded.\n")

# Excluded columns receipt
excluded = [
    ("trend_direction", "label source -- sibling of is_declining_label"),
    ("trend_pct", "numeric parent of trend_direction -- deliberate leak test above"),
    ("is_declining_label", "the target itself"),
    ("impressions_last_30d / prev_30d (and siblings)", "trend-window ingredients"),
    ("content_id / client_id", "pseudonyms -- grouping only, never features"),
]
print("Excluded from X:")
for name, why in excluded:
    print(f"  - {name}: {why}")

# --- Real failure examples on honest holdout ---
te = df.iloc[test_cli].reset_index(drop=True).copy()
te["y"] = honest_scores["y_te"]
te["rf_proba"] = honest_scores["proba"]
top20 = te.sort_values("rf_proba", ascending=False).head(20)
fp = top20.loc[top20["y"] == 0].head(3)
fn = te.loc[te["y"] == 1].sort_values("rf_proba").head(3)

print(f"\nRF top-20 declining rate on client holdout: {top20['y'].mean():.3f}")
print(f"False positives in top-20: {int((top20['y'] == 0).sum())}")

print("\n--- False positives (ranked high, label=0) ---")
for i, (_, r) in enumerate(fp.iterrows(), 1):
    print(
        f"{i}. {r['content_id']}: proba={r['rf_proba']:.3f}, "
        f"imp={int(r['impressions_90d'])}, pos={r['avg_position']}, ctr={r['ctr']:.3f}"
    )
    print(
        "   Observed pattern: high demand + middling CTR/position can look at-risk even when "
        "the snapshot proxy is not 'down' -- stable or branded pages."
    )

print("\n--- False negatives (label=1, scored low) ---")
for i, (_, r) in enumerate(fn.iterrows(), 1):
    print(
        f"{i}. {r['content_id']}: proba={r['rf_proba']:.3f}, "
        f"imp={int(r['impressions_90d'])}, pos={r['avg_position']}, ctr={r['ctr']:.3f}"
    )
    print(
        "   Observed pattern: low-volume downs get low scores -- directional for editor capacity "
        "(deprioritize tiny pages), but not proof those pages do not matter."
    )

imp = pd.Series(
    honest_scores["rf"].feature_importances_, index=NUMERIC_FEATURES
).sort_values(ascending=False)
print("\nTop honest importances:")
print(imp.head(6).round(4).to_string())


Leakage hunt (client-holdout test):
  Honest features (14): precision@50=0.9, roc_auc=0.7641, top=avg_position (0.1566)
  + trend_pct (label sibling): precision@50=1.0, roc_auc=1.0, top=trend_pct (0.889)
  Jump: roc_auc +0.2359, precision@50 +0.1000
  Keep the honest number; trend_pct stays excluded.

Excluded from X:
  - trend_direction: label source -- sibling of is_declining_label
  - trend_pct: numeric parent of trend_direction -- deliberate leak test above
  - is_declining_label: the target itself
  - impressions_last_30d / prev_30d (and siblings): trend-window ingredients
  - content_id / client_id: pseudonyms -- grouping only, never features

RF top-20 declining rate on client holdout: 0.950
False positives in top-20: 1

--- False positives (ranked high, label=0) ---
1. content_a1dd3f309e08: proba=0.897, imp=6250, pos=13.3, ctr=0.110
   Observed pattern: high demand + middling CTR/position can look at-risk even when the snapshot proxy is not 'down' -- stable or branded pages.

-

## 4. Claim rewrite

Take the boldest sentence that could have come out of Week-5 / a naive mentor update, and
rewrite it so every word fits the evidence (observed / measured / directional / decision-support).

| Version | Sentence |
|---|---|
| **Too far** | "Our random forest proves which pages will recover after a refresh and predicts Google ranking changes with 90% precision." |
| **Safe rewrite** | "On a client-holdout of the starter snapshot, the random forest **measured** Precision@50 of *X* against a declining-trend **proxy** (test base rate *Y*). That is a **directional**, **decision-support** ranking signal for which pages look worth reviewing first -- not causal evidence that a refresh recovers traffic, and not a claim about Google's algorithm." |

Also rewrite the Week-5 headline gap carefully: beating the CTR-gap baseline on the same holdout
is an **observed** ranking lift on this proxy, not proof the model finds "true" refresh ROI.

In [4]:
p50 = after["precision@50"]
auc = after["roc_auc"]
base = after["base_rate_test"]

too_far = (
    "Our random forest proves which pages will recover after a refresh and "
    "predicts Google ranking changes with 90% precision."
)

safe = (
    f"On a client-holdout of the starter snapshot (test n={after['n_test']:,}, "
    f"{n_test_clients} unseen clients), the random forest measured Precision@50={p50:.3f} "
    f"and ROC AUC={auc:.3f} against the declining-trend proxy "
    f"(test base rate={base:.3f}; random-split Precision@50 was {before['precision@50']:.3f}). "
    f"That is a directional, decision-support ranking signal for which pages look worth "
    f"reviewing first -- not causal evidence that a refresh recovers traffic, and not a claim "
    f"about Google's algorithm."
)

print("TOO FAR (do not ship):")
print(" ", too_far)
print("\nSAFE REWRITE (keep):")
print(" ", safe)

audit = {
    "paper_findings_questioned": [
        {
            "finding": "RF Precision@50 ~0.74 vs baseline ~0.24 (~3x lift)",
            "question": "Label is snapshot trend proxy -- does P@50 support refresh-first claims?",
        },
        {
            "finding": "Client-holdout presented as transfer evidence",
            "question": "With 32 clients, how sensitive is lift to holdout membership / time split?",
        },
    ],
    "before_after_split": comparison.to_dict(orient="records"),
    "gap_random_minus_client": {
        "precision@50": round(gap_p50, 4),
        "roc_auc": round(gap_auc, 4),
    },
    "leakage_attack": {
        "honest": {k: honest_scores[k] for k in ["precision@50", "roc_auc", "top_feature", "top_importance"]},
        "with_trend_pct": {k: leaky_scores[k] for k in ["precision@50", "roc_auc", "top_feature", "top_importance"]},
    },
    "safe_claim": safe,
    "claim_language": ["observed", "measured", "directional", "decision-support"],
}
AUDIT_PATH.write_text(json.dumps(audit, indent=2), encoding="utf-8")
print(f"\nWrote {AUDIT_PATH}")


TOO FAR (do not ship):
  Our random forest proves which pages will recover after a refresh and predicts Google ranking changes with 90% precision.

SAFE REWRITE (keep):
  On a client-holdout of the starter snapshot (test n=2,325, 6 unseen clients), the random forest measured Precision@50=0.900 and ROC AUC=0.764 against the declining-trend proxy (test base rate=0.391; random-split Precision@50 was 0.960). That is a directional, decision-support ranking signal for which pages look worth reviewing first -- not causal evidence that a refresh recovers traffic, and not a claim about Google's algorithm.

Wrote work\outputs\w06_validation_audit.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.